# Fine-tuning Notebook: LiquidAI LFM2.5-350M-Base

Ce notebook prépare un fine-tuning supervisé `LoRA + TRL` pour `LiquidAI/LFM2.5-350M-Base` à partir des fichiers `data/liquid/liquid_train.jsonl` et `data/liquid/liquid_eval.jsonl`.

Sources officielles utilisées :

- Liquid Docs, TRL: https://docs.liquid.ai/customization/finetuning-frameworks/trl
- Liquid Docs, Dataset formats: https://docs.liquid.ai/docs/fine-tuning/datasets
- Hugging Face model card: https://huggingface.co/LiquidAI/LFM2.5-350M-Base

Notes :

- Le modèle `Base` n'est pas instruction-tuned. Ici, on l'adapte via SFT sur un dataset au format `messages`.
- La stratégie recommandée dans la doc Liquid est `LoRA` plutôt qu'un full fine-tuning.
- Les chemins sont déjà alignés avec la structure actuelle du projet.

In [1]:
%pip install -q "trl>=0.9.0" "transformers>=4.55.0" "datasets>=2.20.0" "accelerate>=0.34.0" "peft>=0.12.0" torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 18.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.8 MB/s eta 0:00:00:00:0100:01


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
TRAIN_PATH = PROJECT_ROOT / "data" / "liquid" / "liquid_train.jsonl"
EVAL_PATH = PROJECT_ROOT / "data" / "liquid" / "liquid_eval.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "lfm25_350m_lora"
MODEL_ID = "LiquidAI/LFM2.5-350M-Base"

print("train:", TRAIN_PATH)
print("eval :", EVAL_PATH)
print("out  :", OUTPUT_DIR)
assert TRAIN_PATH.exists(), f"Missing: {TRAIN_PATH}"
assert EVAL_PATH.exists(), f"Missing: {EVAL_PATH}"

train: /content/data/liquid/liquid_train.jsonl
eval : /content/data/liquid/liquid_eval.jsonl
out  : /content/artifacts/lfm25_350m_lora


AssertionError: Missing: /content/data/liquid/liquid_train.jsonl

In [ ]:
import json
from datasets import load_dataset

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_preview = json.loads(next(line for line in f if line.strip()))

print(train_preview.keys())
print([m["role"] for m in train_preview["messages"]])
print(train_preview["messages"][0]["content"][:200])

In [ ]:
dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_PATH),
        "eval": str(EVAL_PATH),
    },
)

dataset

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

sample_messages = dataset["train"][0]["messages"]
rendered = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
)

print(rendered[:1500])

## Hyperparamètres recommandés

Pour un `350M`, on reste volontairement simple :

- `LoRA` au lieu de full fine-tuning
- `learning_rate=2e-4` comme point de départ raisonnable pour LoRA
- `max_length=1024` pour limiter la mémoire et garder un apprentissage stable
- `gradient_checkpointing=True` pour réduire la consommation mémoire
- `bf16=True` si ton GPU le supporte ; sinon remplace par `fp16=True`

In [ ]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    gradient_checkpointing=True,
    bf16=True,
    max_length=1024,
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=tokenizer,
    peft_config=peft_config,
)

trainer

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
metrics = trainer.evaluate()
metrics

In [ ]:
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved adapter and tokenizer to: {OUTPUT_DIR}")

In [ ]:
import torch

test_messages = [
    {
        "role": "system",
        "content": "Tu es un analyste VC et Lean Startup. Tu réponds de façon courte et priorisée.",
    },
    {
        "role": "user",
        "content": (
            "Instruction: Analyse ce passage et classe-le correctement.\n"
            "Input: Pour une startup SaaS B2B, nous avons beaucoup de démos mais la rétention à 30 jours chute fortement, "
            "et pourtant l'équipe veut accélérer le recrutement commercial.\n"
            "Label attendu: Red Flag\n"
            "Severity attendue: Fatal\n"
            "Difficulty attendue: Implicite"
        ),
    },
]

prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.2,
        top_k=50,
        repetition_penalty=1.05,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Remarques pratiques

- Si `bf16=True` ne passe pas sur ta machine, remplace par `fp16=True`.
- Si la mémoire GPU est trop juste, baisse `per_device_train_batch_size` à `4` ou `2`.
- Si tu veux une évaluation plus difficile, remplace `liquid_eval.jsonl` par `data/liquid/liquid_hard_eval.jsonl`.
- Si tu veux pousser vers Hugging Face Hub, ajoute ensuite `trainer.push_to_hub()` avec ton auth.